# A minimal agent: a model, two tools and a loop

An agent is a language model that can act, not only answer. This notebook builds the smallest one that does
something real: it reads your GPT-2 lab results on Drive and answers questions about them. Every piece is here in
plain Python, so you can see what each one does and why it exists.

| Piece | What it is | Why it's needed |
|---|---|---|
| Model | Qwen3-1.7B, instruction-tuned | Decides the next step: answer, or call a tool. Its architecture is the lab's `qwen3-style`, scaled up |
| Tools | plain Python functions: `list_files`, `read_file` | The model can only produce text. Tools let that text cause actions and bring back facts it doesn't know |
| Tool schema | each tool's name, description and arguments, put in the prompt | The model can only call what it has been told exists, with the arguments it has been told it takes |
| Loop | generate → if it asked for a tool, run it, append the result, generate again | One generation can't both ask for a file and read it. The loop turns single replies into multi-step work |
| Memory | the message list, which becomes the context window | Everything the agent knows lives here. This is what the recall test measured |

The trace prints every step: what the model wrote, which tool ran, what came back, and how full the context is.

**On Colab:** Runtime → Change runtime type → **T4** or **L4** → Run all. The model needs about 4 GB of GPU memory.
To use the fine-tuned model from `agent_finetune`, set `USE_ADAPTER = True` in the config cell.

In [ ]:
# Config.
import os, sys
IN_COLAB = 'google.colab' in sys.modules
MODEL = os.environ.get('MODEL', 'Qwen/Qwen3-1.7B')
LAB = os.environ.get('AGENT_ROOT', '/content/drive/MyDrive/gpt2-lab' if IN_COLAB else '.')   # the only folder tools can see
OUT = os.environ.get('AGENT_OUT', '/content/drive/MyDrive/agent-lab' if IN_COLAB else 'out')
USE_ADAPTER = False      # True: load the LoRA adapter trained by agent_finetune from OUT/finetune/adapter
MAX_STEPS = 8            # model replies before the loop gives up: an agent that never stops is the classic failure
MAX_NEW_TOKENS = 768     # per model reply
THINKING = False         # Qwen3 can reason in a <think> block before replying: better plans, many more tokens

In [ ]:
# Setup: the model and its tokenizer.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if dev == 'cpu' or torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # T4: fp16
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype, device_map=dev if dev == 'cuda' else None).eval()
if USE_ADAPTER:
    if IN_COLAB:
        import subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft'], check=True)
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, f'{OUT}/finetune/adapter').eval()
print(f'{MODEL}{" + LoRA adapter" if USE_ADAPTER else ""} on {dev} · '
      f'{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters')

## The agent core: tools, tool-call parsing and the loop
Shared by `mini_agent` and `agent_finetune` (build_notebooks.py pastes this file into both), so the fine-tuned model
is trained on exactly the prompt it will see when used.

In [ ]:
# Tools. Plain Python functions. The type hints and the docstring are not decoration: transformers turns them into the
# JSON schema that goes into the prompt, and that schema is all the model knows about each tool.
import os, re, json

ROOT = os.path.realpath('.')
READ_LIMIT = 4000        # characters a single read_file returns: every result stays in the context for good

def set_root(path):
    """Point the tools at a folder. They cannot see anything outside it."""
    global ROOT
    ROOT = os.path.realpath(path)

def safe_path(path: str) -> str:
    """Keep every tool inside ROOT, so the model can't read anything else on the machine."""
    full = os.path.realpath(os.path.join(ROOT, path))
    if os.path.commonpath([full, ROOT]) != ROOT:
        raise ValueError(f'{path} is outside the allowed folder')
    return full

def list_files(folder: str = '.') -> str:
    """
    List the files and subfolders in a folder, with file sizes.

    Args:
        folder: Path relative to the lab folder, for example "." or "recall".
    """
    full = safe_path(folder)
    out = []
    for name in sorted(os.listdir(full)):
        p = os.path.join(full, name)
        out.append(f'{name}/' if os.path.isdir(p) else f'{name}  ({os.path.getsize(p):,} bytes)')
    return '\n'.join(out) or '(empty folder)'

def read_file(path: str) -> str:
    """
    Read a text file (csv, json, md, txt, py). Long files are cut off.

    Args:
        path: Path relative to the lab folder, for example "recall/summary.csv".
    """
    text = open(safe_path(path), encoding='utf-8', errors='replace').read()
    return text if len(text) <= READ_LIMIT else text[:READ_LIMIT] + f'\n... [cut: {len(text):,} characters in total]'

def sort_table(path: str, column: str, descending: bool = True) -> str:
    """
    Sort a CSV table by one numeric column and list every run with its value, in order. Use it to find the highest or
    lowest value instead of comparing numbers by eye.

    Args:
        path: Path to a .csv file relative to the lab folder, for example "results/summary.csv".
        column: The column to sort by, written exactly as in the table's header.
        descending: true lists the highest value first, false the lowest first.
    """
    import csv
    if isinstance(descending, str):
        descending = descending.strip().lower() not in ('false', '0', 'no')
    with open(safe_path(path), encoding='utf-8', errors='replace', newline='') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return f'Error: {path} has no rows'
    header = list(rows[0])
    if column not in header:
        return f'Error: there is no column {column!r} in {path}. Columns: {", ".join(header)}'
    key = next((h for h in ('run', 'name', 'model') if h in header), header[0])
    def value(r):
        try:
            return float(r[column])
        except (TypeError, ValueError):
            return None
    have = sorted((r for r in rows if value(r) is not None), key=value, reverse=descending)
    blank = len(rows) - len(have)
    return (f'{path} sorted by {column}, {"highest" if descending else "lowest"} first '
            f'({len(have)} rows{f", {blank} blank" if blank else ""}):\n{key},{column}\n'
            + '\n'.join(f'{r[key]},{r[column]}' for r in have))

TOOLS = {f.__name__: f for f in (list_files, read_file, sort_table)}

def run_tool(name, args):
    """Run one tool call. Errors go back to the model as text, so it can correct itself instead of crashing the loop."""
    if name not in TOOLS:
        return f'Error: there is no tool named {name}. Available: {", ".join(TOOLS)}'
    try:
        return TOOLS[name](**args)
    except Exception as e:
        return f'Error: {type(e).__name__}: {e}'

In [ ]:
# The model's side of tool calling. Qwen3 was trained to answer a tool request in one fixed format:
#   <tool_call>
#   {"name": "read_file", "arguments": {"path": "recall/summary.csv"}}
#   </tool_call>
# The chat template writes the tool schemas into the prompt; parse_tool_calls reads that format back out.
TOOL_CALL = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.S)

def parse_tool_calls(text):
    calls = []
    for m in TOOL_CALL.finditer(text):
        try:
            c = json.loads(m.group(1))
            calls.append(dict(name=c['name'], arguments=c.get('arguments', {})))
        except (json.JSONDecodeError, KeyError, TypeError) as e:
            calls.append(dict(name='invalid', arguments={}, error=f'Error: could not parse {m.group(1)!r}: {e}'))
    return calls

def assistant_turn(reply, calls):
    """The message stored for a reply that called tools. The chat template writes each call from `tool_calls`, so the
    <tool_call> text comes out of `content`: left in, every call would appear twice in the history."""
    return dict(role='assistant', content=TOOL_CALL.sub('', reply).strip(),
                tool_calls=[dict(type='function', function=dict(name=c['name'], arguments=c['arguments'])) for c in calls])

In [ ]:
# The loop. This is the whole agent. `generate(messages)` returns (reply, prompt length in tokens).
SYSTEM = ('You are a research assistant for a lab that trains small language models and benchmarks them. '
          'Answer from the files, not from memory: list folders to find files, read them, then answer. '
          'For the highest or lowest value in a table, use sort_table. '
          'Quote the numbers you use and name the file they came from.')

def run_agent(task, generate, max_steps=8, verbose=True):
    messages = [dict(role='system', content=SYSTEM), dict(role='user', content=task)]
    n_calls = bad_calls = 0
    for step in range(1, max_steps + 1):
        reply, ctx = generate(messages)
        calls = parse_tool_calls(reply)
        if verbose:
            print(f'\n── step {step} · context {ctx:,} tokens ──\n{reply.strip()}')
        if not calls:                                         # no tool requested: this is the final answer
            return dict(answer=reply.strip(), steps=step, calls=n_calls, bad_calls=bad_calls, context=ctx, messages=messages)
        messages.append(assistant_turn(reply, calls))
        for c in calls:
            result = c['error'] if 'error' in c else run_tool(c['name'], c['arguments'])
            n_calls += 1
            bad_calls += result.startswith('Error')
            if verbose:
                print(f'  → {c["name"]}({json.dumps(c["arguments"])}) returned {len(result):,} characters')
            messages.append(dict(role='tool', name=c['name'], content=result))
    return dict(answer=None, steps=max_steps, calls=n_calls, bad_calls=bad_calls, context=ctx, messages=messages)

In [ ]:
set_root(LAB)

@torch.no_grad()
def generate(messages):
    prompt = tok.apply_chat_template(messages, tools=list(TOOLS.values()), add_generation_prompt=True,
                                     tokenize=False, enable_thinking=THINKING)
    ids = tok(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
    out = model.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    reply = tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True)
    return reply, ids['input_ids'].shape[1]

In [ ]:
# Try it. Change the task to anything about the files in LAB.
result = run_agent('In the recall test results, which model copies best when the first copy is far back, '
                   'and how does attn-gqa compare with llama-style?', generate, max_steps=MAX_STEPS)
print('\n══ answer ══\n' + str(result['answer']))

## What to look for in the trace
- **The first step is a tool call, not an answer.** The system prompt and the tool schemas steered it to look before
  answering. Remove the tools (`TOOLS.clear()`) and it will answer from memory: fluent and wrong.
- **Context grows every step.** Each tool result is appended and stays. At 4,000 characters per read that's about
  1,000 tokens per file, so a 20-step task can reach 16K+ tokens: the range the Qwen3 recall test checks.
- **Errors are fed back, not raised.** Ask it to read a file that doesn't exist and watch it recover by listing the
  folder first.
- **Where a 1.7B model breaks:** malformed tool calls, re-reading the same file, or misreading a table. The untuned
  model's first run (baseline_trace_qwen3-1.7b.txt) called the tools correctly, then restated the whole table instead
  of answering. `agent_finetune` trains that out and measures the change.